In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
import re, collections
import seaborn as sns

ACCENT, GREY, LIGHT = "#1F6F8B", "#9AA8B3", "#C9D6DC"
PAIR = [ACCENT, GREY]
sns.set_theme(style="whitegrid", context="notebook", rc={"figure.figsize": (8, 4.2),
              "axes.spines.top": False, "axes.spines.right": False})

def hbar(data, y, x, ax, fmt="{:,.0f}", color=ACCENT):
    """Horizontal seaborn bar chart in the frame's order, with value labels."""
    sns.barplot(data=data, y=y, x=x, ax=ax, color=color, orient="h", order=list(data[y]))
    ax.bar_label(ax.containers[0], labels=[fmt.format(v) for v in data[x]], padding=3, fontsize=8)
    ax.set_xlim(0, data[x].max() * 1.18)
    ax.set_ylabel("")

def paired_share(data, cat, cols, names, ax):
    """Two percentage columns side by side per category (e.g. % repos vs % files)."""
    long = data.melt(id_vars=cat, value_vars=cols, var_name="measure", value_name="pct")
    long["measure"] = long.measure.map(dict(zip(cols, names)))
    sns.barplot(data=long, x=cat, y="pct", hue="measure", palette=PAIR, ax=ax, order=list(data[cat]))
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.1f", fontsize=8)
    ax.set_ylabel("%"); ax.legend(title="")

from pathlib import Path

def _abs_parquet(rel):
    """Resolve a path relative to src/notebooks and return the absolute parquet path."""
    here = Path.cwd().resolve()
    data = None
    for candidate in (here, *here.parents):
        if (candidate / 'src' / 'data').is_dir() and (candidate / 'src' / 'notebooks').is_dir():
            data = candidate / 'src' / 'data'
            break
        if candidate.name == 'notebooks' and (candidate.parent / 'data').is_dir():
            data = (candidate.parent / 'data').resolve()
            break
    if data is None:
        raise FileNotFoundError(f'Could not locate src/data from {here}')
    path = (data / rel.removeprefix('../data/')).resolve()
    matches = list(path.parent.glob(path.name)) if '*' in path.name else ([path] if path.is_file() else [])
    if not matches:
        raise FileNotFoundError(path)
    return str(path)

repos_path = _abs_parquet('../data/gitskills_data/data/repos/*.parquet')
artifacts_path = _abs_parquet('../data/gitskills_data/data/artifacts/*.parquet')
artifact_siblings_path = _abs_parquet('../data/gitskills_data/data/artifact_siblings/*.parquet')
mining_runs_path = _abs_parquet('../data/gitskills_data/data/mining_runs/*.parquet')
skill_features_path = _abs_parquet('../data/generated_data/skill_features.parquet')
file_agents_path = _abs_parquet('../data/generated_data/file_agents.parquet')

con = duckdb.connect(database=':memory:')

## Agent folders

Where skill files sit in agent-specific folders, and how those folders are combined.

Each file is labeled by the first dot-folder in its path (`.claude`, `.cursor`, `.github`, and so on). The summaries below use `file_agents.parquet`, which holds that label for every artifact.


In [7]:
files_by_agent = con.execute(f"""
SELECT agent_folder,
       COUNT(*)                                        AS files,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_files,
       COUNT(DISTINCT repo_full_name)                  AS repos,
       COUNT(DISTINCT file_sha)                        AS distinct_skills
FROM '{file_agents_path}'
GROUP BY 1 ORDER BY files DESC;
""").df()
files_by_agent


,agent_folder,files,pct_files,repos,distinct_skills
0,No agent folder,2370702,62.4,150627,1305436
1,Claude Code,431630,11.4,68467,305831
2,Shared (.agents),324605,8.5,44274,184008
3,Other dot-folder,151743,4.0,10939,69115
4,opencode,129554,3.4,4947,71279
5,Shared (.agent),117346,3.1,5825,36332
6,GitHub Copilot,91670,2.4,15044,61814
7,Cursor,48352,1.3,8146,33754
8,Codex,45784,1.2,6046,32607
9,Gemini CLI,29274,0.8,1975,13005


In [8]:
folders_per_repo = con.execute(f"""
WITH r AS (
  SELECT repo_full_name, COUNT(DISTINCT agent_folder) AS n_folders
  FROM '{file_agents_path}'
  WHERE agent_folder <> 'No agent folder'
  GROUP BY 1
)
SELECT n_folders, COUNT(*) AS repos,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_repos
FROM r GROUP BY 1 ORDER BY 1;
""").df()
folders_per_repo


,n_folders,repos,pct_repos
0,1,128893,88.3
1,2,12992,8.9
2,3,2571,1.8
3,4,713,0.5
4,5,300,0.2
5,6,122,0.1
6,7,75,0.1
7,8,80,0.1
8,9,53,0.0
9,10,44,0.0


In [9]:
agent_combos = con.execute(f"""
WITH r AS (
  SELECT repo_full_name, string_agg(DISTINCT agent_folder, ' + ' ORDER BY agent_folder) AS combo,
         COUNT(DISTINCT agent_folder) AS n
  FROM '{file_agents_path}'
  WHERE agent_folder <> 'No agent folder'
  GROUP BY 1
)
SELECT combo, COUNT(*) AS repos
FROM r WHERE n > 1
GROUP BY 1 ORDER BY repos DESC LIMIT 15;
""").df()
agent_combos


,combo,repos
0,Claude Code + Shared (.agents),5820
1,Claude Code + Codex,826
2,Claude Code + Other dot-folder,657
3,Claude Code + GitHub Copilot,609
4,Claude Code + Cursor,500
5,Other dot-folder + Shared (.agents),488
6,GitHub Copilot + Shared (.agents),452
7,Cursor + Shared (.agents),330
8,Shared (.agent) + Shared (.agents),304
9,Claude Code + Other dot-folder + Shared (.agents),294


In [ ]:
mirroring = con.execute(f"""
WITH m AS (
  SELECT repo_full_name, file_sha, COUNT(DISTINCT agent_folder) AS n_folders
  FROM '{file_agents_path}'
  GROUP BY 1, 2
)
SELECT COUNT(*)                                   AS repo_skill_pairs,
       COUNT_IF(n_folders > 1)                    AS mirrored_pairs,
       ROUND(100.0 * COUNT_IF(n_folders > 1) / COUNT(*), 1) AS pct_mirrored,
       COUNT(DISTINCT CASE WHEN n_folders > 1 THEN repo_full_name END) AS repos_with_mirroring
FROM m;
""").df()
mirroring.T.rename(columns={0: "value"})


In [ ]:
mirrored_folder_pairs = con.execute(f"""
WITH f AS (
  SELECT DISTINCT repo_full_name, file_sha, agent_folder FROM '{file_agents_path}'
)
SELECT a.agent_folder AS folder_a, b.agent_folder AS folder_b, COUNT(*) AS mirrored_skills
FROM f a JOIN f b
  ON a.repo_full_name = b.repo_full_name AND a.file_sha = b.file_sha
 AND a.agent_folder < b.agent_folder
GROUP BY 1, 2 ORDER BY mirrored_skills DESC LIMIT 15;
""").df()
mirrored_folder_pairs


In [ ]:
agents = con.execute(f"""
    SELECT agent_folder,
           COUNT(DISTINCT repo_full_name) AS repos,
           COUNT(*)                       AS files
    FROM '{file_agents_path}'
    WHERE agent_folder <> 'No agent folder'
    GROUP BY 1
    ORDER BY repos DESC
""").df()

fig, ax = plt.subplots(figsize=(9, 5.5))
y = range(len(agents))[::-1]                      # largest at the top
bars = ax.barh(list(y), agents.repos, color="lightblue")
ax.set_yticks(list(y), agents.agent_folder)
ax.bar_label(bars, labels=[f"{r:,}" for r in agents.repos], padding=3, fontsize=8)
ax.set_xlabel("Repositories")
ax.set_xlim(0, agents.repos.max() * 1.15)
ax.set_title("Repositories with skills in each agent folder")
plt.tight_layout(); plt.show()
agents

## Agent ecosystem by path

**Where skills live.** Most skill files (62.4%, 2.37M) sit outside any agent's
own folder, in plain `skills/` directories, plugin trees and collections. Among
agent folders, Claude Code's `.claude/` is the most widespread (431,630 files in
68,467 repositories), followed by the tool-neutral `.agents/` convention
(324,605 files, 44,274 repositories). GitHub Copilot (15,044 repositories),
Cursor (8,146), Codex (6,046) and opencode (4,947) follow at a distance, and
each remaining tool (Gemini CLI, Kiro, Windsurf, Trae, Qwen Code, Kilo Code,
Continue, Factory) appears in under 2,000 repositories. `SKILL.md`, introduced
by Anthropic, is thus used across the agent ecosystem, and a shared
tool-neutral location is already the second most common home.

**Files per repository vary by tool.** Claude Code averages 6.3 skill files per
repository, while opencode averages 26 and Antigravity 116 (13,458 files in
only 116 repositories). High ratios indicate collection or installer
repositories that bundle hundreds of skills, rather than projects adding a few.

**Multi-agent repositories.** Of 145,894 repositories with at least one agent
folder, 88.3% use exactly one; 11.7% (17,001) use two or more, and a few use up
to 16. The dominant combination is Claude Code + `.agents/` (5,820
repositories), followed by Claude Code paired with Codex (826), Copilot (609)
or Cursor (500). Claude Code is part of nearly every common combination, so
multi-agent support typically extends a Claude Code setup to other tools.

**Mirroring.** Only 3.1% of repository–skill pairs (96,270) place the same
skill text in two or more agent folders, across 12,367 repositories. The most
mirrored pair is Claude Code ↔ `.agents/` (30,831 skills), then Claude Code
with Cursor (7,895) and Codex (7,194). Mirroring is a real practice for
supporting several agents, but it explains only a small share of the in-repo
duplication found earlier; the rest comes from copies within the same folder
tree, such as plugin packs and nested project templates.

**Caveats.** Folders are identified by the first dot-folder in a path, so nested
layouts (e.g. `.config/opencode/`) fall under "other dot-folder"; `.github/`
also holds non-Copilot files, so its count is an upper bound for Copilot.